# OOD Main3 Consistency Ablation: Single-Source OOD TF-IDF Baselines

Scenario filter: `single_source_ood`

This notebook reads the scenario-specific Main3 consistency ablation bundles and summarizes the TF-IDF baseline feature sets only. It prefers `Baseline: TF-IDF last sentence` plus `Baseline: TF-IDF full prefix`, and falls back to `Baseline: TF-IDF prefix` if the full-prefix baseline is not present in the selected results root.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "ood_main3_consistency_scenario_tables_lib.py").exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import ood_main3_consistency_scenario_tables_lib as scenario_tables


scenario_tables = importlib.reload(scenario_tables)


In [ ]:
RESULTS_ROOT = Path(
    "/playpen-ssd/smerrill/deception2/Results/OOD_Modeling_main3_consistency_xgb_only_tfidf2"
)
# Example alternate path for a rerun that includes full_prefix_text TF-IDF features:
# RESULTS_ROOT = Path(
#     "/playpen-ssd/smerrill/deception2/Results/OOD_Modeling_main3_consistency_xgb_only_tfidf_full_prefix"
# )

RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
print(f"Using results root: {RESULTS_ROOT}")


In [ ]:
SCENARIO_NAME = "single_source_ood"
SCENARIO_TITLE = "Train on 1 environment; evaluate OOD on the other 4"

_, _, _, preview_metrics_df = scenario_tables.load_scenario_bundle_frames(
    RESULTS_ROOT,
    SCENARIO_NAME,
)

available_feature_sets = (
    set(preview_metrics_df["feature_set"].dropna().astype(str).tolist())
    if "feature_set" in preview_metrics_df.columns
    else set()
)

selected_feature_order = []
if "Baseline: TF-IDF last sentence" in available_feature_sets:
    selected_feature_order.append("Baseline: TF-IDF last sentence")
if "Baseline: TF-IDF full prefix" in available_feature_sets:
    selected_feature_order.append("Baseline: TF-IDF full prefix")
elif "Baseline: TF-IDF prefix" in available_feature_sets:
    selected_feature_order.append("Baseline: TF-IDF prefix")

if not selected_feature_order:
    selected_feature_order = [
        "Baseline: TF-IDF last sentence",
        "Baseline: TF-IDF full prefix",
    ]

requested_feature_sizes = scenario_tables.infer_requested_feature_sizes(preview_metrics_df) or [32]

print(f"Using baseline feature order: {selected_feature_order}")
print(f"Using requested feature sizes: {requested_feature_sizes}")


In [ ]:
inventory_df, panel_df, model_df, metrics_df = scenario_tables.render_scenario_notebook(
    scenario_name=SCENARIO_NAME,
    scenario_title=SCENARIO_TITLE,
    results_root=RESULTS_ROOT,
    requested_feature_sizes=requested_feature_sizes,
    show_heatmaps=False,
    save_heatmaps=False,
    feature_order=selected_feature_order,
    feature_group_specs=[("TF-IDF baselines", selected_feature_order)],
    include_attention_ablation=False,
)
